# Experiments
This file will evaluate the results of the synthetic dataset with the ground truth

In [ ]:
import os
import sys
sys.path.insert(0, '../')
import drm

## Object Detection


In [ ]:
from pathlib import Path

binPath = Path("/home/jvermandere/projects/DRM/_input/virtualDataset/VirtualScanner-1773153754053/main.bin")#"/home/jvermandere/projects/DRM/_input/Office2.bin"
# Define the output directory
bb_detected = (binPath.parent / "results" / "preds" / binPath.stem).with_suffix(".json")
bb_gt = binPath.as_posix()[:-4] + "_bb.txt"

### Check the bounding boxes

In [ ]:
import trimesh
import numpy as np
import json

with open(bb_detected) as f:
    data = json.load(f)

# Parameters
score_threshold = 0.5  # Only show boxes with score > threshold

# Colormap for labels
label_colors = [
    [1, 0, 0, 0.5],  # red, alpha 0.5
    [0, 1, 0, 0.5],  # green
    [0, 0, 1, 0.5],  # blue
    [1, 1, 0, 0.5],  # yellow
    [1, 0, 1, 0.5],  # magenta
    [0, 1, 1, 0.5],  # cyan
]

# Create list of meshes
bb_meshes = []

for label, score, box in zip(data["labels_3d"], data["scores_3d"], data["bboxes_3d"]):
    if score < score_threshold:
        continue

    center = np.array(box[:3])
    size = np.array(box[3:6])
    rotation_z = box[6]
    color = label_colors[label % len(label_colors)]
    
    mesh = drm.create_trimesh_box(center, size, rotation_z, color)
    bb_meshes.append(mesh)

# Combine meshes for visualization
scene = trimesh.Scene(bb_meshes)
pcd = drm.load_bin_pointcloud(str(binPath))
scene.add_geometry(pcd)

# Show interactive visualization
scene.show()

### ground truth boxes

In [ ]:
import json
import numpy as np

def load_boxes(json_path):
    with open(json_path) as f:
        data = json.load(f)

    names = []
    points_list = []

    for box in data["boxes"]:
        names.append(box["id"])
        pts = np.array([[p["x"], p["y"], p["z"]] for p in box["boundingPoints"]])
        points_list.append(pts)

    return points_list, names

gt_boxes, gt_names = load_boxes(bb_gt)

# Colormap for labels
label_colors = [
    [1, 0, 0, 0.5],  # red, alpha 0.5
    [0, 1, 0, 0.5],  # green
    [0, 0, 1, 0.5],  # blue
    [1, 1, 0, 0.5],  # yellow
    [1, 0, 1, 0.5],  # magenta
    [0, 1, 1, 0.5],  # cyan
]


# Create list of meshes
gt_bb_meshes = []
i = 0
for points in gt_boxes:
    color = label_colors[i % len(label_colors)]
    
    mesh = trimesh.convex.convex_hull(points)
    mesh.visual.face_colors = color
    mesh.apply_transform(drm.UNITY2TRIMESH_T)
    gt_bb_meshes.append(mesh)
    i+=1

# Combine meshes for visualization
scene = trimesh.Scene(gt_bb_meshes)
pcd = drm.load_bin_pointcloud(str(binPath))
scene.add_geometry(pcd)

# Show interactive visualization
scene.show()

#mesh = trimesh.convex.convex_hull(vertices)

### Both Boxes

In [ ]:
# Combine meshes for visualization
scene = trimesh.Scene(bb_meshes)
scene.add_geometry(gt_bb_meshes)

# Show interactive visualization
scene.show()

### IOU calculation

In [ ]:
import numpy as np
import trimesh
from scipy.optimize import linear_sum_assignment


def mesh_iou(mesh_a, mesh_b):
    inter = trimesh.boolean.intersection([mesh_a, mesh_b])
    if inter is None or inter.volume == 0:
        return 0.0

    union = trimesh.boolean.union([mesh_a, mesh_b])
    return inter.volume / union.volume


def compute_iou_matrix(pred_meshes, gt_meshes):
    iou_matrix = np.zeros((len(pred_meshes), len(gt_meshes)))

    for i, p in enumerate(pred_meshes):
        for j, g in enumerate(gt_meshes):
            iou_matrix[i, j] = mesh_iou(p, g)

    return iou_matrix


def evaluate_meshes(pred_meshes, gt_meshes, iou_threshold=0.5):

    iou_matrix = compute_iou_matrix(pred_meshes, gt_meshes)

    # Hungarian matching (maximize IoU)
    pred_idx, gt_idx = linear_sum_assignment(-iou_matrix)

    matches = []
    tp = 0
    matched_ious = []

    for p, g in zip(pred_idx, gt_idx):
        iou = iou_matrix[p, g]
        matches.append((p, g, iou))

        if iou >= iou_threshold:
            tp += 1
            matched_ious.append(iou)

    fp = len(pred_meshes) - tp
    fn = len(gt_meshes) - tp

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    avg_iou = np.mean(matched_ious) if matched_ious else 0

    return {
        "precision": precision,
        "recall": recall,
        "average_iou": avg_iou,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "matches": matches
    }
metrics = evaluate_meshes(bb_meshes, gt_bb_meshes, 0.5)

print("Precision:", metrics["precision"])
print("Recall:", metrics["recall"])
print("Average IoU:", metrics["average_iou"])

for p, g, iou in metrics["matches"]:
    print(f"Pred {p} ↔ GT {g}  IoU={iou:.3f}")

## Scene Completion

In [ ]:
emptyScenePly = "/home/jvermandere/projects/DRM/_input/virtualDataset/VirtualScanner-1773153754053/results/segmented_points/isolated_points_reconstructed.ply"
gtSceneBin = "/home/jvermandere/projects/DRM/_input/virtualDataset/VirtualScanner-1773153754053/main_empty.bin"

import trimesh
import sys
sys.path.insert(0, '../')
import drm

pcd = trimesh.load(emptyScenePly)
gt_pcd = drm.load_bin_pointcloud(str(gtSceneBin))

In [ ]:
trimesh.Scene([pcd,gt_pcd]).show()

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
import trimesh

def chamfer_distance(pc1: trimesh.points.PointCloud, 
                     pc2: trimesh.points.PointCloud) -> float:
    """
    Computes the symmetric Chamfer distance between two point clouds.

    Args:
        pc1, pc2: trimesh.points.PointCloud objects

    Returns:
        chamfer_dist: float, sum of mean squared distances both ways
    """
    # Build KDTree for fast nearest neighbor queries
    tree_pc2 = cKDTree(pc2.vertices)
    tree_pc1 = cKDTree(pc1.vertices)

    # For each point in pc1, find closest point in pc2
    d1, _ = tree_pc2.query(pc1.vertices, k=1)
    # For each point in pc2, find closest point in pc1
    d2, _ = tree_pc1.query(pc2.vertices, k=1)

    # Chamfer distance = sum of mean squared distances both ways
    chamfer_dist = np.mean(d1**2) + np.mean(d2**2)
    return chamfer_dist

In [ ]:
cd = chamfer_distance(pcd, gt_pcd)
print("Chamfer distance:", cd)

## Object Completion
The obejct completions are evaluated using the chamfer distance